# Get embeddings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import streamlit as st
import pandas as pd
from pathlib import Path
from final_project_package.embeddings.embeddings import load_clip_model, get_text_embeddings, get_similarity

In [3]:
model, processor = load_clip_model()
images_df = pd.read_csv(Path.cwd() / "data_dump/images_cleaned_embedding.csv", nrows=1000)
images_df = images_df[images_df["embedding"] != "[]"]
listings_df = pd.read_csv(Path.cwd() / "data_dump/listings_with_scores.csv")
source_id = images_df["source_id"]
listings_df = listings_df[listings_df["source_id"].isin(source_id)]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
len(images_df)

331

In [6]:
query = "expensive toilet"

variable = list(listings_df.columns)[0]

min_value = 0.0

text_embedding = get_text_embeddings(model, processor, [query])
type(text_embedding)

torch.Tensor

In [7]:
%time
similarity = images_df["embedding"].apply(lambda x: \
        get_similarity(x, text_embedding))

similarity.head()

CPU times: user 5 μs, sys: 2 μs, total: 7 μs
Wall time: 11.2 μs


/home/bea/code/katiarojas87/final-project/final_project_package/embeddings/embeddings.py:21: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  similarity = tensor(eval(image_embedding)) @ tensor(text_embedding.T).numpy()
/home/bea/code/katiarojas87/final-project/final_project_package/embeddings/embeddings.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  similarity = tensor(eval(image_embedding)) @ tensor(text_embedding.T).numpy()


45    tensor(0.1958)
46    tensor(0.1992)
47    tensor(0.2776)
58    tensor(0.2334)
59    tensor(0.2076)
Name: embedding, dtype: object

In [8]:
if query == "":
    listings = listings_df
else:
    source_id = images_df[similarity > 0.2]["source_id"]
    listings = listings_df[listings_df["source_id"].isin(source_id)]

df = pd.DataFrame({
    "lon": listings["longitude"],
    "lat": listings["latitude"]
})

df.head()

,lon,lat
4,139.739929,35.718056
6,139.866165,35.762589
10,139.660233,35.718670
11,139.628342,35.738468
12,139.629211,35.736996
